# Day 4: Structured Streaming (Simulation)

## Objective
Simulate a streaming data pipeline using the H2 weather data.

## What this covers
* Micro-batch streaming from a landing folder
* Checkpointing for fault tolerance
* Writing streaming output to Delta
* Querying streaming results in real-time

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql import functions as F
import time, os, shutil

CATALOG = "dbacademy"
SCHEMA  = "labuser13955035_1772876383"
VOL     = f"/Volumes/{CATALOG}/{SCHEMA}/h2files"

# Streaming paths
STREAM_INPUT  = f"/tmp/h2_stream_input"
STREAM_OUTPUT = f"{CATALOG}.{SCHEMA}.weather_stream_delta"
CHECKPOINT    = f"/tmp/h2_stream_checkpoint"

print(f"Stream input:  {STREAM_INPUT}")
print(f"Stream output: {STREAM_OUTPUT}")
print(f"Checkpoint:    {CHECKPOINT}")

In [0]:
# Clean up previous runs
for path in [STREAM_INPUT, CHECKPOINT]:
    dbutils.fs.rm(path, recurse=True)
dbutils.fs.mkdirs(STREAM_INPUT)
spark.sql(f"DROP TABLE IF EXISTS {STREAM_OUTPUT}")

# Load weather source and split into 5 micro-batches
weather_df = spark.read.csv(f"{VOL}/era5_nl_2020_hourly.csv", header=True, inferSchema=True)
total = weather_df.count()
batch_size = total // 5

print(f"Source: {total:,} rows → 5 batches of ~{batch_size:,} rows")

# Write batch 1 immediately so streaming has data to start with
batch1 = weather_df.limit(batch_size)
batch1.write.mode("overwrite").option("header", True).csv(f"{STREAM_INPUT}/batch_001")
print(f"\n✅ Batch 1 written ({batch_size} rows) - streaming can start")

In [0]:
# Define schema explicitly (required for streaming from files)
weather_schema = spark.read.csv(f"{VOL}/era5_nl_2020_hourly.csv", header=True, inferSchema=True).schema

# Read stream from landing folder
stream_df = (
    spark.readStream
    .schema(weather_schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)  # Process 1 file per micro-batch
    .csv(STREAM_INPUT)
)

# Add ingestion metadata
stream_enriched = stream_df.withColumn(
    "ingested_at_utc", F.current_timestamp()
)

# Write stream to Delta table
query = (
    stream_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .toTable(STREAM_OUTPUT)
)

print(f"✅ Streaming started → {STREAM_OUTPUT}")
print(f"   Query ID: {query.id}")
print(f"   Status:   {query.status}")

In [0]:
# Simulate new data arriving in 4 more batches
weather_pd = weather_df.toPandas()

for i in range(2, 6):
    start = (i - 1) * batch_size
    end = min(i * batch_size, total)
    batch = spark.createDataFrame(weather_pd.iloc[start:end])
    batch.write.mode("overwrite").option("header", True).csv(f"{STREAM_INPUT}/batch_{i:03d}")
    print(f"📥 Batch {i} arrived ({end - start} rows)")
    time.sleep(5)  # Wait for stream to pick up

print("\n✅ All 5 batches delivered")
time.sleep(10)  # Let final batch process

In [0]:
# Check how many rows the stream has processed
result_count = spark.table(STREAM_OUTPUT).count()
print(f"\n📊 Streaming Delta table: {result_count:,} rows")
print(f"   Source had:            {total:,} rows")

# Show latest ingested data
print("\nLatest records:")
display(
    spark.table(STREAM_OUTPUT)
    .orderBy(F.col("ingested_at_utc").desc())
    .limit(5)
)

In [0]:
# Stop the streaming query
query.stop()
print("✅ Streaming query stopped")

# Show Delta version history - each batch = 1 version
print("\n📜 Delta version history (each batch = 1 commit):")
display(spark.sql(f"DESCRIBE HISTORY {STREAM_OUTPUT}"))

In [0]:
# Clean up temp files
dbutils.fs.rm(STREAM_INPUT, recurse=True)
dbutils.fs.rm(CHECKPOINT, recurse=True)
print("✅ Temp files cleaned up")
print("\n🎉 Day 4 Structured Streaming complete!")